# Engine equivalence test: multi-color Moran process

Runs the **same graphs**, the **same number of trials**, on both engines, and asks whether
every node comes out with the same **probability of founding the winning lineage** and the
same **time to takeover**.

| section | what it does |
|---|---|
| 1. Setup and cost | graphs, trial count, and what the run will cost |
| 2. Run both engines | one `run_repeats` call per (graph, engine) |
| 3. Test: per-node win probability | two-proportion z-test, per node |
| 4. Test: per-node takeover time | two-sample KS test, per node |
| 5. Verdict | Bonferroni + p-value uniformity, one PASS/FAIL line |
| 6. See it | per-node profiles, scatter, and the graphs drawn side by side |
| 7. Positive control | the same tests on deliberately biased data, to show they can fail |

### What "the same" means here

The two engines are **statistically equivalent, not bit-exact**. The C++ core draws from
xoshiro256++ and the Python reference from NumPy's PCG64, so trial 7 of a given seed is a
different trajectory in each. What must match is the *distribution*: run enough trials and
every per-node number converges to the same value.

So this notebook cannot check `python_result == cpp_result`. It checks that the two samples
are consistent with having come from one distribution, which is a hypothesis test, and
**a high p-value is the passing outcome**. This is the same standard, and the same pair of
tests, that `scripts/compare_batches.py` applies to the two-color engine.

### Ground truth

There is also an exact answer to check both engines against. Under this update rule
(uniform reproducer, uniform neighbor overwritten) a node's neutral fixation probability is
proportional to **1/deg(v)**. A high-degree node reproduces no more often than anyone else
but is overwritten constantly, since every neighbor that reproduces may target it. The
graphs below are chosen to span that: `complete` and `cycle` are regular, so 1/deg is flat
and the answer is 1/N; `star` is the extreme, with a hub of degree 11 among leaves of
degree 1.

> **Run this in an `ijup` or `inode` session.** The Python engine leg is a few minutes of
> sustained single-core work, and the login node's watchdog kills CPU-heavy processes.

---

## 1. Setup and cost

In [ ]:
%load_ext autoreload
%autoreload 2
%cd /home/labs/pilpel/matanyaw/moran-process

import sys

sys.path.insert(0, "src")

import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from moran_process import CppMultiColorMoranProcess, MultiColorMoranProcess, PopulationGraph
from moran_process.pipeline import multicolor_batch as mcb

# The two things under test, keyed by the name the pipeline uses for them.
ENGINES = {"python": MultiColorMoranProcess, "cpp": CppMultiColorMoranProcess}

In [ ]:
N_TRIALS = 1_000
MAX_STEPS = 1_000_000
SEED = 20260831
ALPHA = 0.05  # family-wise, Bonferroni-corrected across all tests below

# Small graphs on purpose: the Python reference has to run every one of these
# 10,000 times, and it is ~1500x slower than the core. N=12-15 keeps that to
# minutes while still spanning the full range of 1/deg behaviour.
GRAPHS = [
    PopulationGraph.complete_graph(12),      # regular -> 1/deg is flat at 1/N
    PopulationGraph.cycle_graph(12),         # regular -> also flat, different mixing
    PopulationGraph.line_graph(12),          # endpoints deg 1: 2x the interior
    PopulationGraph.star_graph(12),          # hub deg 11: the extreme case
    PopulationGraph.mammalian_lung_graph(),  # a real respiratory topology
    PopulationGraph.random_connected_graph(12)
]

for g in GRAPHS:
    core = g.to_simulation_struct()
    deg = np.diff(core.offsets)
    print(f"{g.name:18} N={g.number_of_nodes():3d}  degrees {deg.min()}..{deg.max()}")

`preview` runs a handful of real trials on each graph and extrapolates. Run it before the
real thing, because the Python leg is the whole cost of this notebook and consensus time is
set by how long lineages take to **meet**, not by node count.

In [ ]:
print("--- python engine (the expensive leg) ---")
cost_py = mcb.preview(GRAPHS, n_trials=N_TRIALS, pilot=30, max_steps=MAX_STEPS, engine="python")
print("\n--- cpp engine ---")
cost_cpp = mcb.preview(GRAPHS, n_trials=N_TRIALS, pilot=300, max_steps=MAX_STEPS, engine="cpp")

---

## 2. Run both engines

Each (graph, engine) pair is one `run_repeats` call, which is a single crossing into the
engine rather than `N_TRIALS` of them.

Two details that matter for fairness:

* The graph is converted to its CSR `GraphCore` **once** and both engines are handed that
  same object, so node ordering and neighbor lists are identical by construction rather
  than by coincidence.
* Every leg gets its own seed, drawn from one root generator. The engines produce
  different streams from the same seed anyway, so reusing one would not make the samples
  paired. Independent seeds make that explicit.

In [ ]:
seeds = np.random.default_rng(SEED).integers(0, 2**31, size=(len(GRAPHS), len(ENGINES)))

frames, timing = [], []
for gi, g in enumerate(GRAPHS):
    core = g.to_simulation_struct()  # built once, shared by both engines
    for ei, (engine, Engine) in enumerate(ENGINES.items()):
        sim = Engine(core, max_steps=MAX_STEPS, seed=int(seeds[gi, ei]))
        t0 = time.perf_counter()
        out = sim.run_repeats(N_TRIALS)
        elapsed = time.perf_counter() - t0

        frames.append(
            pd.DataFrame(
                {
                    "graph": g.name,
                    "engine": engine,
                    "winner": out["winner"],
                    "fixed": out["fixed"],
                    "steps": out["steps"],
                }
            )
        )
        timing.append(
            {
                "graph": g.name,
                "N": core.n_nodes,
                "engine": engine,
                "seconds": elapsed,
                "M_steps_per_s": out["steps"].sum() / elapsed / 1e6,
                "n_censored": int((~out["fixed"]).sum()),
            }
        )
        print(f"  {g.name:18} {engine:6} {elapsed:8.2f}s")

raw = pd.concat(frames, ignore_index=True)
speed = pd.DataFrame(timing)
print(f"\n{len(raw):,} trials total")

A censored trial (one that hit `max_steps` without reaching consensus) has `winner = -1`
and no node to credit, so it cannot enter either test. At these sizes there should be none;
the assertion below makes that a fact rather than an assumption.

In [ ]:
assert speed["n_censored"].sum() == 0, (
    f"{speed['n_censored'].sum()} censored trials. Raise MAX_STEPS: a censored run has no "
    "winner and is silently absent from both tests.\n"
    f"{speed[speed.n_censored > 0].to_string(index=False)}"
)

wide = speed.pivot(index="graph", columns="engine", values="seconds")
wide["speedup"] = wide["python"] / wide["cpp"]
wide["python_M_steps_s"] = speed[speed.engine == "python"].set_index("graph")["M_steps_per_s"]
wide["cpp_M_steps_s"] = speed[speed.engine == "cpp"].set_index("graph")["M_steps_per_s"]
print(wide.to_string(float_format=lambda v: f"{v:.3g}"))
print(f"\ntotal: python {wide['python'].sum():.1f}s   cpp {wide['cpp'].sum():.2f}s"
      f"   overall speedup {wide['python'].sum() / wide['cpp'].sum():.0f}x")

### The per-node table

One row per (graph, node, engine). `win_frac` is the node's share of wins;
`mean_steps` is the takeover time **conditional on that node winning**, which is the
per-node time the question asks about.

`n_wins` is worth reading before the time results: it is the sample size behind that node's
`mean_steps`. On the star, the hub wins about 0.8% of trials, so its time estimate rests on
~80 runs and is correspondingly noisy in both engines. That is a property of the graph, not
a defect in either implementation.

In [ ]:
recs = []
for g in GRAPHS:
    core = g.to_simulation_struct()
    deg = np.diff(core.offsets).astype(float)
    theory = (1.0 / deg) / (1.0 / deg).sum()  # the 1/deg law, normalised
    for engine in ENGINES:
        sub = raw[(raw["graph"] == g.name) & (raw["engine"] == engine) & raw["fixed"]]
        winner = sub["winner"].to_numpy()
        steps = sub["steps"].to_numpy()
        n_fixed = len(sub)
        wins = np.bincount(winner, minlength=core.n_nodes)
        for v in range(core.n_nodes):
            s_v = steps[winner == v]
            recs.append(
                {
                    "graph": g.name,
                    "node": v,
                    "engine": engine,
                    "deg": int(deg[v]),
                    "theory_prob": theory[v],
                    "n_fixed": n_fixed,
                    "n_wins": int(wins[v]),
                    "win_frac": wins[v] / n_fixed,
                    "sem_frac": np.sqrt(wins[v] / n_fixed * (1 - wins[v] / n_fixed) / n_fixed),
                    "mean_steps": s_v.mean() if len(s_v) else np.nan,
                    "sem_steps": stats.sem(s_v) if len(s_v) > 1 else np.nan,
                }
            )

node_df = pd.DataFrame(recs)
node_df.head(12)

---

## 3. Test: per-node win probability

Two-proportion z-test per node, the same function `scripts/compare_batches.py` uses on the
two-color engine's fixation probability. It asks whether `wins_python / n` and
`wins_cpp / n` are consistent with one underlying probability.

High p is the passing outcome.

In [ ]:
def two_proportion_z(k1, n1, k2, n2):
    '''Two-sided z-test for equality of two proportions; returns (z, p).

    Copied from scripts/compare_batches.py so the multi-color engine is judged
    by exactly the standard the two-color engine was.
    '''
    p_pool = (k1 + k2) / (n1 + n2)
    se = np.sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))
    if se == 0:
        return 0.0, 1.0
    z = (k1 / n1 - k2 / n2) / se
    return z, 2 * (1 - stats.norm.cdf(abs(z)))


piv = node_df.pivot(index=["graph", "node", "deg", "theory_prob"], columns="engine")
rows = []
for (gname, v, deg, theory), r in piv.iterrows():
    z, p = two_proportion_z(
        r[("n_wins", "python")], r[("n_fixed", "python")],
        r[("n_wins", "cpp")], r[("n_fixed", "cpp")],
    )
    rows.append(
        {
            "graph": gname, "node": v, "deg": deg, "theory_prob": theory,
            "frac_python": r[("win_frac", "python")],
            "frac_cpp": r[("win_frac", "cpp")],
            "z_prob": z, "p_prob": p,
        }
    )
prob_cmp = pd.DataFrame(rows)

print("Per-graph summary (max |z| over that graph's nodes, and the smallest p):\n")
print(
    prob_cmp.groupby("graph")
    .agg(nodes=("node", "size"), max_abs_z=("z_prob", lambda s: s.abs().max()),
         min_p=("p_prob", "min"))
    .to_string(float_format=lambda v: f"{v:.4g}")
)

---

## 4. Test: per-node takeover time

Two-sample Kolmogorov-Smirnov test per node, on the **full distribution** of step counts for
the trials that node won, not just the mean. Same choice as the two-color comparison:
fixation times are heavy-tailed, so two samples can share a mean and still come from
visibly different distributions, and the KS test sees that where a t-test would not.

In [ ]:
rows = []
for gname, gdf in raw.groupby("graph"):
    for v in sorted(gdf.loc[gdf["fixed"], "winner"].unique()):
        a = gdf[(gdf.engine == "python") & gdf.fixed & (gdf.winner == v)]["steps"].to_numpy()
        b = gdf[(gdf.engine == "cpp") & gdf.fixed & (gdf.winner == v)]["steps"].to_numpy()
        if len(a) < 2 or len(b) < 2:
            rows.append({"graph": gname, "node": v, "n_python": len(a), "n_cpp": len(b),
                         "mean_python": np.nan, "mean_cpp": np.nan,
                         "ks_stat": np.nan, "p_time": np.nan})
            continue
        ks = stats.ks_2samp(a, b)
        rows.append(
            {
                "graph": gname, "node": v, "n_python": len(a), "n_cpp": len(b),
                "mean_python": a.mean(), "mean_cpp": b.mean(),
                "ks_stat": ks.statistic, "p_time": ks.pvalue,
            }
        )
time_cmp = pd.DataFrame(rows)

print("Per-graph summary (smallest p over that graph's nodes):\n")
print(
    time_cmp.groupby("graph")
    .agg(nodes=("node", "size"), min_wins=("n_python", "min"),
         max_ks=("ks_stat", "max"), min_p=("p_time", "min"))
    .to_string(float_format=lambda v: f"{v:.4g}")
)

---

## 5. Verdict

Both tests run once per node per graph, so there are many p-values and a raw 0.05 threshold
would flag several of them by chance even if the engines were identical. Two guards, again
matching `compare_batches.py`:

* **Bonferroni.** A single test counts as a failure only below `ALPHA / n_tests`, which
  controls the chance of *any* false flag across the whole family.
* **p-value uniformity.** Bonferroni is deliberately blunt, and a small systematic bias
  would push every p-value down a little without tripping it. Under true equivalence the
  p-values are uniform on (0, 1), so a KS test of the collection against Uniform(0,1)
  catches exactly that: many mildly low p-values with no single dramatic one.

Both must pass.

In [ ]:
def verdict(prob_cmp, time_cmp, alpha=ALPHA, label="python vs cpp"):
    '''Print the family-wise verdict for one pair of comparison frames.'''
    p_prob = prob_cmp["p_prob"].dropna().to_numpy()
    p_time = time_cmp["p_time"].dropna().to_numpy()
    all_p = np.concatenate([p_prob, p_time])
    n_tests = len(all_p)
    bonf = alpha / n_tests

    flagged_prob = prob_cmp[prob_cmp["p_prob"] < bonf]
    flagged_time = time_cmp[time_cmp["p_time"] < bonf]
    unif_p = stats.kstest(all_p, "uniform").pvalue

    bonf_ok = len(flagged_prob) == 0 and len(flagged_time) == 0
    unif_ok = unif_p > alpha
    passed = bonf_ok and unif_ok

    print(f"{label}")
    print(f"  tests run            : {n_tests}  ({len(p_prob)} probability + {len(p_time)} time)")
    print(f"  Bonferroni threshold : alpha/{n_tests} = {bonf:.2e}")
    print(f"  flagged below it     : {len(flagged_prob)} probability, {len(flagged_time)} time"
          f"   -> {'ok' if bonf_ok else 'FAIL'}")
    print(f"  p-value uniformity   : KS vs Uniform(0,1) p = {unif_p:.4f}"
          f"   -> {'ok' if unif_ok else 'FAIL'}")
    print(f"  smallest p seen      : {all_p.min():.2e}   (median {np.median(all_p):.3f}, "
          "expected ~0.5)")
    if not bonf_ok:
        print("\n  flagged rows:")
        for f in (flagged_prob, flagged_time):
            if len(f):
                print(f.to_string(index=False))

    bar = "=" * 68
    print(f"\n{bar}")
    if passed:
        print("VERDICT: PASS - the two engines are statistically indistinguishable")
    else:
        print("VERDICT: FAIL - the engines differ by more than Monte Carlo error")
    print(bar)
    return passed, all_p


passed, all_p = verdict(prob_cmp, time_cmp)

---

## 6. See it

### Per-node profiles

The direct picture: for every graph, each node's win fraction under both engines, with the
1/deg law drawn through them. Error bars are +/- 1 standard error, so points agreeing to
within their bars is the expected appearance of a pass.

Read `complete` and `cycle` against `star` and `line`. The first two are regular, so 1/deg
is flat and every node sits at 1/N. The star's hub is an order of magnitude below its
leaves, and both engines have to reproduce that gap, not just each other.

In [ ]:
n_g = len(GRAPHS)
fig, axes = plt.subplots(1, n_g, figsize=(4.0 * n_g, 3.8))
axes = np.atleast_1d(axes)  # subplots returns a bare Axes when n_g == 1
colors = {"python": "#e6194b", "cpp": "#4363d8"}

for ax, g in zip(axes, GRAPHS):
    sub = node_df[node_df["graph"] == g.name]
    theory = sub[sub["engine"] == "python"].sort_values("node")
    ax.plot(theory["node"], theory["theory_prob"], "-", color="0.35", lw=1.6,
            label="1/deg (exact)", zorder=1)
    for engine, c in colors.items():
        e = sub[sub["engine"] == engine].sort_values("node")
        ax.errorbar(e["node"], e["win_frac"], yerr=e["sem_frac"], fmt="o", ms=5,
                    color=c, alpha=0.85, capsize=2, label=engine, zorder=2)
    ax.axhline(1 / g.number_of_nodes(), color="0.75", ls=":", lw=1.2,
               label="uniform 1/N", zorder=0)
    ax.set_title(g.name, fontsize=10)
    ax.set_xlabel("node")
    ax.set_yscale("log")
axes[0].set_ylabel("win fraction (log scale)")
axes[0].legend(fontsize=7, loc="best")
fig.suptitle(f"Per-node win probability, {N_TRIALS:,} trials per engine", y=1.02)
plt.tight_layout()
plt.show()

### One engine against the other

Every node of every graph, plotted python on x against cpp on y. Perfect agreement puts
every point on the dashed diagonal; the error bars say how far off the diagonal a point is
entitled to sit.

The time panel is the looser of the two, and necessarily so: `mean_steps` for a node rests
only on the trials **that node won**, roughly `N_TRIALS / N` of them, so it carries several
percent of noise where the probability estimate carries a fraction of one percent.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5.4))
_palette = ["#4363d8", "#3cb44b", "#f58231", "#911eb4", "#e6194b", "#469990", "#9a6324"]
gcolors = {g.name: _palette[i % len(_palette)] for i, g in enumerate(GRAPHS)}

for gname, sub in prob_cmp.groupby("graph"):
    nd = node_df[node_df["graph"] == gname]
    sx = nd[nd["engine"] == "python"].sort_values("node")["sem_frac"].to_numpy()
    sy = nd[nd["engine"] == "cpp"].sort_values("node")["sem_frac"].to_numpy()
    s = sub.sort_values("node")
    ax1.errorbar(s["frac_python"], s["frac_cpp"], xerr=sx, yerr=sy, fmt="o", ms=6,
                 color=gcolors[gname], alpha=0.85, capsize=2, label=gname)

t = time_cmp.dropna(subset=["mean_python"])
for gname, sub in t.groupby("graph"):
    nd = node_df[node_df["graph"] == gname]
    sx = nd[nd["engine"] == "python"].sort_values("node")["sem_steps"].to_numpy()
    sy = nd[nd["engine"] == "cpp"].sort_values("node")["sem_steps"].to_numpy()
    s = sub.sort_values("node")
    ax2.errorbar(s["mean_python"], s["mean_cpp"], xerr=sx, yerr=sy, fmt="o", ms=6,
                 color=gcolors[gname], alpha=0.85, capsize=2, label=gname)

for ax, title, xlabel in (
    (ax1, "Per-node win probability", "win fraction, python engine"),
    (ax2, "Per-node takeover time (conditional on that node winning)", "mean steps, python engine"),
):
    lo = min(ax.get_xlim()[0], ax.get_ylim()[0])
    hi = max(ax.get_xlim()[1], ax.get_ylim()[1])
    ax.plot([lo, hi], [lo, hi], "k--", lw=1.2, alpha=0.6, zorder=0, label="y = x")
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_aspect("equal")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(xlabel)
ax1.set_ylabel("win fraction, cpp engine")
ax2.set_ylabel("mean steps, cpp engine")
ax1.legend(fontsize=7, loc="upper left")
plt.tight_layout()
plt.show()

### The p-value distribution

Under true equivalence these are uniform on (0, 1), and the histogram should look flat and
featureless. A pile-up against zero is what a real difference looks like, and it is what
section 7 deliberately produces.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.hist(all_p, bins=20, range=(0, 1), color="#4363d8", alpha=0.8, edgecolor="white")
ax.axhline(len(all_p) / 20, color="crimson", ls="--", lw=1.5, label="uniform expectation")
ax.set_xlabel("p-value")
ax.set_ylabel("count")
ax.set_title(f"All {len(all_p)} p-values  (KS vs Uniform(0,1): p = "
             f"{stats.kstest(all_p, 'uniform').pvalue:.3f})")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### The graphs themselves, side by side

The same measurement as the profile plot, drawn onto the topology with
`PopulationGraph.draw_colored_graph`. Both panels share one color scale, so a real
difference between the engines would show up as a visible difference in shading between
left and right.

In [ ]:
SHOW = "star_n12"  # try "line_n12" or "mammalian_b2_d3"

g = next(x for x in GRAPHS if x.name == SHOW)
sub = node_df[node_df["graph"] == SHOW]
expected = 1.0 / g.number_of_nodes()
vals = {e: sub[sub["engine"] == e].sort_values("node")["win_frac"].to_numpy() for e in ENGINES}

# One shared scale across both panels, wide enough for the worse of the two, so
# the comparison is not flattered by autoscaling each panel to its own range.
half = max(np.abs(np.concatenate(list(vals.values())) - expected).max(), 1e-12)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, (engine, v) in zip(axes, vals.items()):
    g.draw_colored_graph(
        v, cmap="RdYlGn", label="win fraction", center=expected, half_width=half,
        node_size=500, ax=ax, with_labels=True,
        title=f"{SHOW} - {engine} engine ({N_TRIALS:,} trials)",
    )
plt.tight_layout()
plt.show()

print(sub.pivot(index="node", columns="engine",
                values=["win_frac", "mean_steps"]).to_string(float_format=lambda x: f"{x:.4g}"))

---

## 7. Positive control: can these tests fail?

A test that passes on everything proves nothing. This section re-runs both tests against a
deliberately corrupted copy of the cpp results, to confirm they have the power to notice a
difference at this trial count.

* **Probability**: move 1.5% of the win mass onto node 0, taken proportionally from the rest.
* **Time**: stretch every step count by 5%.

Both perturbations are small enough to be invisible on the plots above, and the expected
result is **FAIL**. Watch *which guard catches which*, because they do different jobs:

* The probability shift concentrates its damage on one node per graph, so it produces a
  handful of individually extreme p-values and **Bonferroni** catches it.
* The 5% time stretch is spread thinly over every node, and at 10,000 trials no single
  node's KS test crosses the corrected threshold. It is caught only by the
  **p-value uniformity** check, which sees the collective downward drift. This is the case
  Bonferroni alone would wave through, and the reason both guards are here.

Power depends on `N_TRIALS`. If you lowered it to make the notebook quick, this control may
legitimately pass, which means the run was too small to resolve differences of this size,
not that the tests are broken. The cell below says so if it happens.

In [ ]:
BIAS_PROB = 0.015  # win-mass shifted onto node 0
BIAS_TIME = 1.05  # multiplicative stretch on every step count

bad_raw = raw.copy()
rng = np.random.default_rng(SEED + 1)
mask = (bad_raw["engine"] == "cpp").to_numpy()

# Time: stretch the cpp step counts.
bad_raw.loc[mask, "steps"] = np.round(bad_raw.loc[mask, "steps"] * BIAS_TIME).astype(int)

# Probability: reassign a small random share of cpp winners to node 0.
idx = np.flatnonzero(mask & (bad_raw["winner"] != 0).to_numpy())
flip = rng.choice(idx, size=int(BIAS_PROB * len(idx)), replace=False)
bad_raw.loc[flip, "winner"] = 0

# Rebuild both comparison frames from the corrupted data, using the same code paths.
bad_prob, bad_time = [], []
for g in GRAPHS:
    core = g.to_simulation_struct()
    gdf = bad_raw[bad_raw["graph"] == g.name]
    counts = {
        e: np.bincount(gdf[(gdf.engine == e) & gdf.fixed]["winner"].to_numpy(),
                       minlength=core.n_nodes)
        for e in ENGINES
    }
    n = {e: int(counts[e].sum()) for e in ENGINES}
    for v in range(core.n_nodes):
        z, p = two_proportion_z(counts["python"][v], n["python"], counts["cpp"][v], n["cpp"])
        bad_prob.append({"graph": g.name, "node": v, "z_prob": z, "p_prob": p})
        a = gdf[(gdf.engine == "python") & gdf.fixed & (gdf.winner == v)]["steps"].to_numpy()
        b = gdf[(gdf.engine == "cpp") & gdf.fixed & (gdf.winner == v)]["steps"].to_numpy()
        if len(a) > 1 and len(b) > 1:
            ks = stats.ks_2samp(a, b)
            bad_time.append({"graph": g.name, "node": v, "ks_stat": ks.statistic,
                             "p_time": ks.pvalue})

bad_passed, bad_all_p = verdict(
    pd.DataFrame(bad_prob), pd.DataFrame(bad_time),
    label=f"python vs cpp+bias  (prob +{BIAS_PROB:.1%} on node 0, time x{BIAS_TIME})",
)
if bad_passed:
    print(
        f"\nNOTE: the control did NOT fail. At N_TRIALS={N_TRIALS:,} this run cannot resolve "
        f"a {BIAS_PROB:.1%} probability shift or a {BIAS_TIME - 1:.0%} time stretch, so section 5's "
        "PASS only rules out differences larger than that. Raise N_TRIALS to tighten it."
    )
else:
    print("\nPositive control failed as intended: the tests above can detect a real difference.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, (p_vals, title) in zip(
    axes,
    [(all_p, f"real comparison  (verdict: {'PASS' if passed else 'FAIL'})"),
     (bad_all_p, f"biased control  (verdict: {'PASS' if bad_passed else 'FAIL'})")],
):
    ax.hist(p_vals, bins=20, range=(0, 1), color="#4363d8", alpha=0.8, edgecolor="white")
    ax.axhline(len(p_vals) / 20, color="crimson", ls="--", lw=1.5)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("p-value")
axes[0].set_ylabel("count")
fig.suptitle("What equivalence looks like, and what a real difference looks like", y=1.03)
plt.tight_layout()
plt.show()